In [1]:
!pip install clearml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 16.5 MB/s eta 0:00:0000:0100:01


## ClearML

В ClearML необходимо тоже авторизоваться. Создайте учетную запись на сайте [https://app.clear.ml/](https://app.clear.ml/).

Затем нужно подключиться к ClearML. Результаты экспериментов будут храниться тоже удаленно.

Строки подключения к ClearML, приведенные ниже можно, получить, перейдя в профиле ClearML -> Settings -> Workspace -> Create new credentials

В kaggle были созданы два секретных ключа `CLEARML_API_ACCESS_KEY` и `CLEARML_API_SECRET_KEY`, значения которых подставляются для авторизации.

In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("CLEARML_API_ACCESS_KEY")
secret_value_1 = user_secrets.get_secret("CLEARML_API_SECRET_KEY")

In [ ]:
# Можно авторизоваться и так
# %env CLEARML_WEB_HOST=https://app.clear.ml/
# %env CLEARML_API_HOST=https://api.clear.ml
# %env CLEARML_FILES_HOST=https://files.clear.ml
# %env CLEARML_API_ACCESS_KEY={secret_value_0}
# %env CLEARML_API_SECRET_KEY={secret_value_1}

In [3]:
import os
os.environ["CLEARML_WEB_HOST"] = "https://app.clear.ml/"
os.environ["CLEARML_API_HOST"] = "https://api.clear.ml"
os.environ["CLEARML_FILES_HOST"] = "https://files.clear.ml"
os.environ["CLEARML_API_ACCESS_KEY"] = secret_value_0
os.environ["CLEARML_API_SECRET_KEY"] = secret_value_1

In [4]:
from clearml import Task
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

class Net(nn.Module):
    def __init__(self, hidden_size, dropout, activation):
        super().__init__()
        self.fc1 = nn.Linear(32*32*3, hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.act = getattr(torch.nn, activation)()
        self.fc2 = nn.Linear(hidden_size, 10)
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.act(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

def train(model, loader, optimizer, criterion, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for data, targets in loader:
        data, targets = data.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(data)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * data.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(targets).sum().item()
        total += targets.size(0)
    return running_loss / total, correct / total

def validate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for data, targets in loader:
            data, targets = data.to(device), targets.to(device)
            outputs = model(data)
            loss = criterion(outputs, targets)
            running_loss += loss.item() * data.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(targets).sum().item()
            total += targets.size(0)
    return running_loss / total, correct / total

In [5]:
# Инициализуем задачу в ClearML
task = Task.init(project_name="cifar10_classification", task_name="Train simple neural model")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
config = {
    'epochs': 10, 'lr': 0.001, 'batch_size': 64,
    'hidden_size': 128, 'dropout': 0.5, 'activation': 'ReLU'
}
task.connect(config)

transform = transforms.ToTensor()
train_set = datasets.CIFAR10(root='.', train=True, download=True, transform=transform)
val_set = datasets.CIFAR10(root='.', train=False, download=True, transform=transform)
train_loader = DataLoader(train_set, batch_size=config['batch_size'], shuffle=True)
val_loader = DataLoader(val_set, batch_size=config['batch_size'])
model = Net(config['hidden_size'], config['dropout'], config['activation']).to(device)
optimizer = optim.Adam(model.parameters(), lr=config['lr'])
criterion = nn.CrossEntropyLoss()
logger = task.get_logger()
for epoch in range(config['epochs']):
    loss, acc = train(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    logger.report_scalar(title="Loss", series="train", value=loss, iteration=epoch)
    logger.report_scalar(title="Acc", series="train", value=acc, iteration=epoch)
    logger.report_scalar(title="Loss", series="val", value=val_loss, iteration=epoch)
    logger.report_scalar(title="Acc", series="val", value=val_acc, iteration=epoch)
    print(f"Epoch {epoch+1}: train loss={loss:.4f}, train acc={acc:.4f}; val loss={val_loss:.4f}, val acc={val_acc:.4f}")

ClearML Task: created new task id=c16c159bd119497fae3e49be7adcc46d
2025-06-07 06:49:26,008 - clearml.Repository Detection - WARNING - Jupyter Notebook auto-logging failed, could not access: /kaggle/working/__notebook_source__.ipynb
2025-06-07 06:49:26,057 - clearml.Task - INFO - Storing jupyter notebook directly as code


2025-06-07 06:49:30.156424: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749278970.377323      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749278970.440215      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


ClearML results page: https://app.clear.ml/projects/cdd9df61dfeb4134a99db0afa1a223cb/experiments/c16c159bd119497fae3e49be7adcc46d/output/log


100%|██████████| 170M/170M [00:15<00:00, 11.1MB/s] 


Epoch 1: train loss=2.1203, train acc=0.2005; val loss=1.9509, val acc=0.2998
Epoch 2: train loss=2.0498, train acc=0.2293; val loss=1.9151, val acc=0.3164
Epoch 3: train loss=2.0340, train acc=0.2324; val loss=1.8816, val acc=0.3133
Epoch 4: train loss=2.0125, train acc=0.2410; val loss=1.8965, val acc=0.3157
Epoch 5: train loss=2.0134, train acc=0.2432; val loss=1.8810, val acc=0.3116
Epoch 6: train loss=2.0116, train acc=0.2413; val loss=1.9064, val acc=0.3067
Epoch 7: train loss=2.0099, train acc=0.2411; val loss=1.8755, val acc=0.3077
Epoch 8: train loss=2.0030, train acc=0.2457; val loss=1.8888, val acc=0.3220
Epoch 9: train loss=2.0054, train acc=0.2428; val loss=1.8680, val acc=0.3241
Epoch 10: train loss=2.0001, train acc=0.2462; val loss=1.8915, val acc=0.3154


In [6]:
task.close()